# Test GLONER (GLiNER + LoRA)

Testing the new GLONER system for model initialization and training.

In [1]:
import sys
sys.path.append('../src')

from models.gloner import GLONER
from config.lora_defaults import DEFAULT_GLINER_MODEL, DEFAULT_LORA_CONFIG, DEFAULT_MAX_LENGTH
from utils.logging import get_logger

logger = get_logger("GLONERTest")
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0" 
print(f"CUDA version: {torch.version.cuda}")
print(f"Number of GPUs visible: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Current GPU: {torch.cuda.current_device()}")
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


CUDA version: 12.8
Number of GPUs visible: 1
Current GPU: 0
GPU Name: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU Memory: 5.7 GB


## 1. Check Defaults

In [3]:
print(F"Default GLiNER model: {DEFAULT_GLINER_MODEL}")
print(f"Default GLiNER model max length: {DEFAULT_MAX_LENGTH}   ")
print("\nDefault LoRA config:")
for key, value in DEFAULT_LORA_CONFIG.items():
    if key == 'target_modules':
        print(f"  {key}: {len(value)} modules")
    else:
        print(f"  {key}: {value}")

Default GLiNER model: knowledgator/modern-gliner-bi-large-v1.0
Default GLiNER model max length: 8192   

Default LoRA config:
  r: 32
  lora_alpha: 64
  lora_dropout: 0.1
  bias: none
  task_type: TaskType.TOKEN_CLS
  target_modules: 18 modules


In [1]:
print("\n" + "="*60)
print("Creating DEFAULT GLONER...")
print("="*60)

# GLONER.default() returns a GLiNER model with default LoRA applied
model = GLONER.default(logger)

print("\n✅ Default GLONER created!")
print(f"Model type: {type(model)}")
print(f"Can use all GLiNER methods: predict_entities, run, evaluate, etc.")


Creating DEFAULT GLONER...


NameError: name 'GLONER' is not defined

## 4. Create Custom GLONER - Custom Model + LoRA + Max Length

In [3]:
print("\n" + "="*60)
print("Creating CUSTOM GLONER (custom model + LoRA + max_length)...")
print("="*60)

print("Pattern for custom model:")

# Use a different GLiNER model with custom LoRA and max_length
custom_full = GLONER.custom(
    logger,
    model_name="urchade/gliner_base",
    max_length=256,
    target_modules=["dense", "query", "key"],
    r=16,
    lora_alpha=32
)


print("\n✅ Pattern shown for fully custom GLONER")


Creating CUSTOM GLONER (custom model + LoRA + max_length)...
Pattern for custom model:


Fetching 4 files: 100%|██████████| 4/4 [01:07<00:00, 16.77s/it]
/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(
/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/torch/cuda/__init__.py:182: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  ret


✅ Pattern shown for fully custom GLONER


In [4]:
display(custom_full)

GLiNER(
  (model): PeftModelForTokenClassification(
    (base_model): LoraModel(
      (model): SpanModel(
        (token_rep_layer): Encoder(
          (bert_layer): Transformer(
            (model): DebertaV2Model(
              (embeddings): DebertaV2Embeddings(
                (word_embeddings): Embedding(128004, 768, padding_idx=0)
                (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
                (dropout): Dropout(p=0.1, inplace=False)
              )
              (encoder): DebertaV2Encoder(
                (layer): ModuleList(
                  (0-11): 12 x DebertaV2Layer(
                    (attention): DebertaV2Attention(
                      (self): DisentangledSelfAttention(
                        (query_proj): Linear(in_features=768, out_features=768, bias=True)
                        (key_proj): Linear(in_features=768, out_features=768, bias=True)
                        (value_proj): Linear(in_features=768, out_features=768, bias=Tru

## 5. Test GLiNER Methods Directly

In [ ]:
print("\n" + "="*60)
print("Testing GLiNER methods directly...")
print("="*60)

# Test text
test_text = ["Apple Inc. is headquartered in Cupertino, California.", "abhishek works at google"]
test_labels = ["organization", "location","person", "actor","genre"]

print(f"\nTest text: {test_text}")
print(f"Labels: {test_labels}")

# GLONER returns a GLiNER model, so use GLiNER methods directly
print("\nPredicting entities...")
entities = custom_full.run(test_text, test_labels)
print(f"\nPredicted entities:")
for entity in entities:
    print(f"  {entity}")

print("\n✅ GLiNER methods working directly on the model!")


Testing GLiNER methods directly...

Test text: ['Apple Inc. is headquartered in Cupertino, California.', 'abhishek works at google']
Labels: ['organization', 'location', 'person']

Predicting entities...

Predicted entities:
  [{'start': 0, 'end': 10, 'text': 'Apple Inc.', 'label': 'organization', 'score': 0.9856753945350647}, {'start': 31, 'end': 40, 'text': 'Cupertino', 'label': 'location', 'score': 0.9908627271652222}, {'start': 42, 'end': 52, 'text': 'California', 'label': 'location', 'score': 0.9670686721801758}]
  [{'start': 0, 'end': 8, 'text': 'abhishek', 'label': 'person', 'score': 0.9571099281311035}, {'start': 18, 'end': 24, 'text': 'google', 'label': 'organization', 'score': 0.9937251210212708}]

✅ GLiNER methods working directly on the model!


## 6. Test Usage Pattern - Training Flow

In [ ]:
print("\n" + "="*60)
print("Training Flow Pattern")
print("="*60)

print('''
# TRAINING FLOW:

from models.gloner import GLONER
from training.trainer import train_lora_model

# 1. Create GLiNER model with LoRA (default or custom)
model = GLONER.default(logger)
# OR
model = GLONER.custom(logger, r=16, lora_alpha=32)

# 2. Train using the trainer
train_lora_model(
    model,
    train_data,
    eval_data,
    training_config,
    save_path="models/my_experiment",
    logger=logger
)

# Result: saves to models/my_experiment/
#   - gliner_model/    (GLiNER checkpoint)
#   - lora_adapter/    (LoRA weights)
#   - checkpoints/     (training checkpoints)
''')

## 7. Test Usage Pattern - Inference Flow

In [13]:
print("\n" + "="*60)
print("Inference Flow Pattern")
print("="*60)


# INFERENCE FLOW:

from models.gloner import GLONER
from data.loader import load_mit_dataset
from evaluation.enchanced_eval import enhanced_evaluate


# Load GLiNER model with trained LoRA adapter
# model = GLONER.load_with_adapter("models/my_experiment/lora_adapter", logger)

test_data, entity_types = load_mit_dataset("../data/mit-movie/test.json", "../data/mit-movie/labels.json")  
print(test_data[0])
print(entity_types)
# OR with custom base model and max_length
model = GLONER.load_with_adapter(
    "../models/active_learning_adapter",
    logger,
    model_name="knowledgator/modern-gliner-bi-large-v1.0",
    max_length=8192
)
display(model)
# Use all GLiNER methods directly
            # Enhanced evaluation on FULL test set
print(device)
model.to(device)
with torch.no_grad():
    gliner_results=model.evaluate(test_data, entity_types, batch_size=8, threshold=0.5 )

            
print(gliner_results)
# The model returned is just a GLiNER model - use it normally!



Inference Flow Pattern
Loading train data from: ../data/mit-movie/test.json
Processed 2442 examples
Entity types: ['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']
{'tokenized_text': ['are', 'there', 'any', 'good', 'romantic', 'comedies', 'out', 'right', 'now'], 'ner': [(4, 5, 'genre'), (7, 8, 'year')]}
['genre', 'year', 'plot', 'average ratings', 'actor', 'title', 'song', 'character', 'rating', 'review', 'director', 'trailer']


Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 137769.11it/s]


GLiNER(
  (model): PeftModelForTokenClassification(
    (base_model): LoraModel(
      (model): SpanModel(
        (token_rep_layer): BiEncoder(
          (bert_layer): Transformer(
            (model): ModernBertModel(
              (embeddings): ModernBertEmbeddings(
                (tok_embeddings): Embedding(50368, 1024, padding_idx=50283)
                (norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
                (drop): Dropout(p=0.0, inplace=False)
              )
              (layers): ModuleList(
                (0): ModernBertEncoderLayer(
                  (attn_norm): Identity()
                  (attn): ModernBertAttention(
                    (Wqkv): lora.Linear(
                      (base_layer): Linear(in_features=1024, out_features=3072, bias=False)
                      (lora_dropout): ModuleDict(
                        (default): Dropout(p=0.1, inplace=False)
                      )
                      (lora_A): ModuleDict(
                   

cuda
('P: 65.80%\tR: 61.07%\tF1: 63.35%\n', np.float64(0.633494905529726))


In [7]:
with torch.no_grad():
    llm_ft_results = enhanced_evaluate( model, test_data, entity_types,threshold=0.5, batch_size=8, has_ground_truth=True)            
print(llm_ft_results)

Running enhanced evaluation...
Processing 2442 examples...
Analyzing errors with ground truth...
{'overall_metrics': {'total_predictions': 5123, 'overall_confidence': np.float64(0.9571452874027843), 'overall_confidence_pct': np.float64(95.71452874027842), 'total_examples': 2442, 'entity_level_accuracy': np.float64(0.5969864581346558), 'entity_level_accuracy_pct': np.float64(59.698645813465575), 'example_level_accuracy': 0.2915642915642916, 'example_level_accuracy_pct': 29.15642915642916, 'overall_f1': np.float64(0.6038973567431989), 'overall_f1_pct': np.float64(60.38973567431989), 'incorrect_examples': [{'tokenized_text': ['are', 'there', 'any', 'good', 'romantic', 'comedies', 'out', 'right', 'now'], 'ner': [(4, 5, 'genre'), (7, 8, 'year')], 'predictions': [[4, 4, 'genre'], [5, 5, 'genre']], 'scores': [0.9999700784683228, 0.8964565396308899], 'errors': {'false_negatives': [[4, 5, 'genre'], [7, 8, 'year']], 'false_positives': [[4, 4, 'genre'], [5, 5, 'genre']]}}, {'tokenized_text': ['sh

In [12]:
llm_ft_results["overall_metrics"]["overall_f1_pct"]

np.float64(60.38973567431989)